# Do logical savings survive hardware compilation?
Superconducting pilot using nine five-qubit inputs from [GuoQ's Nam RZ benchmark suite](https://github.com/qqq-wisc/guoq/tree/main/benchmarks/nam_rz). Run from the project root with the same Python environment and built VOQC library as `comparison.ipynb`.

The baseline is the original input compiled directly. Each candidate first runs VOQC or hardware-unaware Qiskit L3, then uses the **same** backend L1 pipeline as the baseline. Positive values on either axis favor the candidate. These are historical Manila and Lima snapshots.

In [ ]:
from pathlib import Path

from helpers.logical_prediction_helpers import run_pilot, show_pilot_chart

ROOT = Path.cwd()
SUITE = ROOT / "benchmarks" / "nam_rz"
VOQC_HOME = ROOT.parent / "pyvoqc"
OUTPUT = ROOT / "outputs" / "logical_prediction"
CIRCUITS = (
    "barenco_tof_3", "hlf_5", "mod5_4", "muliply_5", "qaoa_5",
    "qe_qft_4", "qe_qft_5", "qft_5", "tof_3",
)
BACKENDS = ("Manila", "Lima")
SEED = 0

## Matched hardware compilation

Both paths use level 1, SABRE, the identity initial placement, seed 0, and exact synthesis. The post-routing hook only observes SWAPs before translation. Compiled circuits are checked against the original unitary with layout accounted for.

Qiskit output permutations are materialized in the logical basis before measuring and compiling. Candidates that fail logical equivalence are excluded and listed above.

The fidelity proxy is the product of calibrated per-instruction fidelities on the mapped qubits. Missing calibration makes the score unavailable. The proxy omits idle noise, crosstalk, and readout. Duration estimates describe device execution of the unitary circuit, excluding queue and classical compilation time.

In [ ]:
points, results, compiled_circuits = run_pilot(
    CIRCUITS, BACKENDS, suite=SUITE, voqc_home=VOQC_HOME, output=OUTPUT, seed=SEED,
)

## Explore the superconducting results

Each mark is one circuit, optimizer, and backend. Hover for the paired values. The zero lines distinguish improvements from regressions. Spearman correlation summarizes the visible points, with average ranks for ties. It is descriptive because the same circuits appear across optimizers and backends. A reversal means the axes have opposite signs.

In [ ]:
show_pilot_chart(points, output=OUTPUT / "chart.html")

## Version 2 · Aer fidelity check

Enable this cell after installing `qiskit-aer` in the notebook environment. It compares the simulated average fidelity of each five-qubit compiled circuit against its noiseless physical-order unitary, using the same backend snapshot for both paths. Gains are in percentage points. The ranking table includes only circuit/backend groups with both optimizers and a complete proxy estimate.

In [ ]:
import pandas as pd
from IPython.display import display

from helpers.logical_prediction_v2 import compare_aer_fidelity

RUN_AER = False

if RUN_AER:
    aer_results, aer_rankings = compare_aer_fidelity(points, compiled_circuits, output=OUTPUT)
    display(aer_rankings)
elif (OUTPUT / "aer_rankings.csv").is_file():
    print("Showing the last saved Aer run. Enable RUN_AER to recompute.")
    display(pd.read_csv(OUTPUT / "aer_rankings.csv"))

## Version 2 · Expanded superconducting benchmark

Enable the run below for 30 circuits, four historical snapshots, and five matched SABRE seeds. Each circuit is compiled on a deterministic connected patch of exactly its width, grown from physical site 0 in breadth-first order. The runner preserves the patch's native gates and calibration and records the physical sites. Set `COMMON_ONLY=True` to compare all selected backends on their shared circuit subset.

The Quartz run uses the bundled upstream Nam 5-gate, 3-qubit equivalence set and a bounded greedy total-gate objective. See [the setup notes](docs/quartz-setup.md) for the pinned source and build steps. Set `QUARTZ_ECC = None` to run just VOQC and Qiskit. The chart reports median paired gains and seed ranges. The last saved run is displayed when recomputation is disabled. Seed-level data and run settings are saved under `OUTPUT / "expanded_quartz_float"` for the three-optimizer run.

In [ ]:
import json

from helpers.logical_prediction_v2 import EXTRA_CIRCUITS, run_expanded_benchmark

RUN_EXPANDED = False
QUARTZ_ECC = OUTPUT / "dependencies/quartz/eccset/Nam_5_3_complete_ECC_set.json"
COMMON_ONLY = False
EXPANDED_OUTPUT = OUTPUT / ("expanded_quartz_float" if QUARTZ_ECC else "expanded")

if RUN_EXPANDED:
    expanded_points = run_expanded_benchmark(
        CIRCUITS + EXTRA_CIRCUITS, ("Manila", "Lima", "Jakarta", "Sherbrooke"),
        (0, 1, 2, 3, 4), suite=SUITE, voqc_home=VOQC_HOME,
        output=EXPANDED_OUTPUT,
        quartz_ecc=QUARTZ_ECC, quartz_seconds=5, common_only=COMMON_ONLY,
    )
    show_pilot_chart(expanded_points, output=EXPANDED_OUTPUT / "chart.html")
elif (EXPANDED_OUTPUT / "expanded_summary.json").is_file():
    print("Showing the last saved expanded run. Enable RUN_EXPANDED to apply changed settings.")
    saved_points = json.loads((EXPANDED_OUTPUT / "expanded_summary.json").read_text())
    show_pilot_chart(saved_points, output=EXPANDED_OUTPUT / "chart.html")